In [4]:
import sys , os
import pandas as pd
import re

sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

# Part 01

In [7]:
file_path = '../data/Sample Messages.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()


cleaned_text = re.sub(r'\\s*', '', raw_text)
messages_list = [line.strip() for line in cleaned_text.split('\n') if line.strip()]

print(f"Total messages to process: {len(messages_list)}\n")


model = pick_model('google', 'general')
client = LLMClient('google', model)


results = []

examples = """
Example 1:
Message: "Breaking News: Kelani River level at 9m."
Output: District: Colombo | Intent: Info | Priority: Low

Example 2:
Message: "We are trapped on the roof with 3 kids!"
Output: District: None | Intent: Rescue | Priority: High

Example 3:
Message: "Does anyone have extra dry rations for the camp in Gampaha?"
Output: District: Gampaha | Intent: Supply | Priority: High

Example 4:
Message: "Just arrived in Galle. Weather is nice."
Output: District: Galle | Intent: Other | Priority: Low
"""


for msg in messages_list[:4]:
    

    prompt_text, spec = render(
        'few_shot.v1',
        role='rescue, supply, info, other classifier',
        examples=examples,
        query=f'Message: "{msg}"',
        constraints='Follow the pattern in examples: provide output in the format "District: <district> | Intent: <intent> | Priority: <priority>". If the district is not mentioned, use "None".',
        format='District: {{district}} | Intent: {{intent}} | Priority: {{priority}}'
    )

    chat_messages = [{'role': 'user', 'content': prompt_text}]
    

    response = client.chat(chat_messages, temperature=0.1) 
    

    output_text = response['text'].strip()
    

    results.append({
        'Original Message': msg,
        'Classification': output_text
    })
    
    print(f"Processed: {msg[:40]}... -> {output_text}")
    

    if 'latency_ms' in response:
        log_llm_call('gemini', model, 'few_shot', response['latency_ms'], response.get('usage', {}))


os.makedirs('../output', exist_ok=True)
df = pd.DataFrame(results)


output_file = '../output/classified_messages.xlsx'
df.to_excel(output_file, index=False)

print(f"\nDone! All results saved to {output_file}")

Total messages to process: 50

Processed: BREAKING: Water levels in Kelani River (... -> District: Colombo | Intent: Info | Priority: High
Processed: SOS: 5 people trapped on a roof in Ja-El... -> District: Gampaha | Intent: Rescue | Priority: High
Processed: Update: Kandy road cleared near Peradeni... -> District: Kandy | Intent: Info | Priority: Low
Processed: Does anyone have extra dry rations for t... -> District: Gampaha | Intent: Supply | Priority: High

Done! All results saved to ../output/classified_messages.xlsx
